# Spine Processing Workflow

This notebook demonstrates the recommended **interactive workflow** for importing and running the MIP file processing functions directly in Jupyter.

## Step 1: Enable Auto-Reload and Import Functions
> **Tip:** `%autoreload 2` ensures that if you edit `copy_mip_file.py` or `batch_copy_mip_files.py`, your notebook picks up the changes automatically without restarting the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import pandas as pd

# Import modular functions from our python scripts
from copy_mip_file import copy_and_rename_file
from batch_copy_mip_files import batch_copy_mip_files

## Step 2: Define Dataset & Output Paths

In [3]:
# Dataset root on network drive / local disk
DATASET_ROOT = r"Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset"

# Destination folder for standardized MIP files
DEST_FOLDER = r"Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset\MIP"

print(f"Source Root : {DATASET_ROOT}")
print(f"Destination : {Path(DEST_FOLDER).resolve()}")

Source Root : Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset
Destination : \\scrps-ishtar.ad.ufl.edu\millerrumbaughlab\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset\MIP


## Step 3: Test a Single File (Sanity Check)
Test copying a single file first to confirm file access, permissions, and correct renaming before running on the entire dataset.

In [4]:
sample_file = r"Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset\Slide12_well2\n1\basal\experiment\timepoint0\Rumbaugh_e19_60X_targetoverview_projection_t0_B01_s0_w0_z0.tif"
sample_name = "Slide12_well2-n1-basal-t0"

# Run single copy function
copy_and_rename_file(sample_file, DEST_FOLDER, sample_name)

Copying:
  Source: Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset\Slide12_well2\n1\basal\experiment\timepoint0\Rumbaugh_e19_60X_targetoverview_projection_t0_B01_s0_w0_z0.tif
  Destination: Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset\MIP\Slide12_well2-n1-basal-t0.tif

[Success] File copied successfully!


WindowsPath('Z:/2026 - E19 Project collab spine and dendrites/Spine/60x_dataset/MIP/Slide12_well2-n1-basal-t0.tif')

## Step 4: Run Batch Processing
Scans the entire `60x_dataset` directory tree, extracts components from path separators, and copies all MIP files to the `MIP/` folder.

In [5]:
# Run batch processing across all folders
batch_copy_mip_files(
    source_root_dir=DATASET_ROOT,
    destination_dir=DEST_FOLDER,
    file_pattern="*projection*.tif"
)

Searching for files matching '*projection*.tif' in:
Z:\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset
Destination directory: \\scrps-ishtar.ad.ufl.edu\millerrumbaughlab\2026 - E19 Project collab spine and dendrites\Spine\60x_dataset\MIP
Found 67 matching file(s).

[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t0_B01_s0_w0_z0.tif
     -> Slide22_well2-n1-basal-t0.tif
[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t0_B01_s0_w0_z0.tif
     -> Slide22_well2-n1-tuft-t0.tif
[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t0_C01_s0_w0_z0.tif
     -> Slide20_well3-n1-basal-t0.tif
[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t1_C01_s0_w0_z0.tif
     -> Slide20_well3-n1-basal-t1.tif
[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t0_A01_s0_w0_z0.tif
     -> Slide14_well1-n3-basal-t0.tif
[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t0_A02_s0_w0_z0.tif
     -> Slide14_well1-n1-basal-t0.tif
[COPIED] Rumbaugh_e19_60X_targetoverview_projection_t0_A02_

## Step 5: Verify & Organize Output in Pandas DataFrame
Load all standardized files in the `MIP` folder into a DataFrame and split the names by delimiters for downstream analysis.

In [ ]:
# List all copied files in destination
mip_files = [f.name for f in Path(DEST_FOLDER).glob("*.tif")]

# Create DataFrame
df_mip = pd.DataFrame({'filename': mip_files})

# Split standardized filename (Slide_well-neuron-region-timepoint.tif) into metadata columns
metadata = df_mip['filename'].str.replace('.tif', '', regex=False).str.split('-', expand=True)
if metadata.shape[1] >= 4:
    metadata.columns = ['slide_well', 'neuron', 'region', 'timepoint']
    df_summary = pd.concat([df_mip, metadata], axis=1)
    display(df_summary.head(10))
else:
    display(df_mip.head(10))
